In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:

from functools import partial
from typing import List, Optional
import jax
import jax.numpy as jnp
import numpy as np

from flax import nnx

from probjax.nn import GaussianFourierEmbedding, Transformer
from probjax.nn.loss_fn.denoising import build_time_dependent_denoising_loss
from probjax.nn.nets.denoising_diffusion_model import EDM
from probjax.nn import MLP
from probjax.nn.utils import AffineFuse
from dmri.simulators import MultiCompartment, AllGaussianModels


In [3]:
AllGaussianModels.fraction_prior

Array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32)

In [4]:
from dmri.nn.tokenizer import DMRITokenizer

In [5]:
dmri_tokenizer = DMRITokenizer(AllGaussianModels, rngs=nnx.Rngs(0))

In [6]:
num_components = len(AllGaussianModels.model_types)
num_total_components = len(AllGaussianModels.model_types) + len(AllGaussianModels.noise_types)

In [7]:
models_mask = jax.random.bernoulli(jax.random.PRNGKey(0), p=0.5, shape=(100, num_total_components))

tokens_cfg = dmri_tokenizer.embed_cfgs(models_mask, jnp.ones((100, num_components)), tuple(AllGaussianModels.model_types), tuple(AllGaussianModels.noise_types), )

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


In [8]:
theta_dim = jnp.sum(AllGaussianModels.split_idx())
thetas = jax.random.normal(jax.random.PRNGKey(0), shape=(100, theta_dim))

In [9]:
tokens_params = jax.jit(dmri_tokenizer.embed_theta)(thetas, tokens_cfg)

In [10]:
tokens_params[0].shape

(19, 64)

In [11]:
dmri_tokenizer.decode_theta(tokens_params)

Array([[ 2.176,  0.433,  1.188, ..., -0.818, -0.058,  0.065],
       [-0.226, -0.311, -0.578, ...,  1.137,  0.085,  0.597],
       [-0.522, -0.075,  0.071, ...,  0.243,  0.356, -0.608],
       ...,
       [ 0.688, -2.436, -0.152, ..., -0.62 , -0.114,  0.719],
       [ 2.793,  0.339,  1.517, ..., -1.458,  0.146, -0.088],
       [ 1.442, -0.62 , -0.019, ...,  1.744, -0.185, -0.851]],      dtype=float32)

In [12]:
dmri_tokenizer.params_dims

(9, 1, 3, 3, 3, 4, 4, 4, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1)

In [83]:
from dmri.nn.dmri_reconstruction_model import DMRIInferenceModel, DMRIInferenceModelConfig

In [84]:
model = DMRIInferenceModel(DMRIInferenceModelConfig(AllGaussianModels), rngs=nnx.Rngs(0))

params = nnx.state(model, nnx.Param)

In [85]:
x = jax.random.normal(jax.random.PRNGKey(0), shape=(100,10))
bvals = jax.random.uniform(jax.random.PRNGKey(0), shape=(100,10), minval=0.1, maxval=1.0)
bvecs = jax.random.normal(jax.random.PRNGKey(0), shape=(100,10,3))
bvecs = bvecs / jnp.linalg.norm(bvecs, axis=-1, keepdims=True)

In [86]:
model(model_mask=models_mask, theta=thetas, x=x, bvals=bvals, bvecs=bvecs)

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)
y (100, 10, 64)
tokens_cfg (100, 19, 64)


(Array([[-1.073,  0.783,  0.551, ..., -0.505,  1.635,  0.6  ],
        [-0.812, -0.85 ,  0.136, ...,  2.068,  2.085,  2.099],
        [-0.485,  0.989,  0.418, ...,  2.306,  2.161,  2.108],
        ...,
        [-0.058,  1.146,  0.849, ...,  0.944,  2.107,  1.222],
        [-0.672, -0.847, -0.792, ...,  1.13 ,  1.373,  1.177],
        [-0.173,  1.143,  0.938, ...,  0.483,  2.71 ,  1.54 ]],      dtype=float32),
 Array([[-1.356, -2.298, -1.581, ..., -0.596,  0.025, -0.081],
        [ 0.851,  0.9  ,  1.999, ...,  0.826,  0.49 , -0.719],
        [ 1.771, -0.48 , -0.2  , ...,  0.176,  2.049,  0.729],
        ...,
        [ 0.153,  1.991,  0.747, ..., -0.452, -0.3  , -0.862],
        [-1.029, -1.778, -2.351, ..., -1.061,  0.84 ,  0.106],
        [-0.913, -1.326,  0.436, ...,  1.267, -0.708,  1.017]],      dtype=float32))

In [88]:
model.loss_fn(params, jax.random.PRNGKey(0), models_mask, thetas, x, bvals, bvecs)

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)


Array([15.668, 15.418], dtype=float32)

In [89]:
# This is rejited
_f = jax.jit(jax.jacrev(model.loss_fn))

In [90]:

grad_per_loss = _f(params, jax.random.PRNGKey(0), models_mask, thetas, x, bvals, bvecs)

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)


In [91]:
grad1 = jax.tree_util.tree_map(lambda x: x[0], grad_per_loss)
grad2 = jax.tree_util.tree_map(lambda x: x[1], grad_per_loss)

In [99]:
jax.tree_util.tree_reduce(lambda x, y: x + y, jax.tree_util.tree_map(lambda x: jnp.sum(jnp.abs(x)), grad1.tokenizer))


Array(266.449, dtype=float32)

In [94]:
jax.tree_util.tree_reduce(
    lambda x, y: x + y,
    jax.tree_util.tree_map(lambda x: jnp.sum(jnp.abs(x)), grad2.tokenizer)
)

Array(497.194, dtype=float32)

In [105]:
def weighted_sum_of_grads(grads, weights):
    """
    grads is a PyTree with shape (T, ...) in each leaf
    weights is a list or array of length T
    Returns a PyTree with the same structure, but shape (...) in each leaf
    """
    def combine_leaf(leaf):
        # leaf has shape (T, ...) — one gradient per task
        return jnp.tensordot(weights, leaf, axes=(0, 0))
        # e.g. sum_i [weights[i] * leaf[i,...]]

    return jax.tree_map(combine_leaf, grads)

def per_task_scaled_norm(grads, alpha, eps=1e-8):
    """
    Compute the L2 norm of alpha_i*g_i for each task i,
    returning shape (T,).
    grads is a PyTree with shape (T, ...) in each leaf.
    alpha is shape (T,).
    """
    # We'll do a vmap over i in [0..T-1].
    def scaled_norm_i(i):
        # For each leaf, pick leaf[i,...] and multiply by alpha[i]
        # Then sum up squares across all leaves/dimensions
        # Return sqrt(...) for L2 norm
        def leaf_contrib(leaf):
            g_i = leaf[i]  # shape (...)
            return jnp.sum((alpha[i] * g_i)**2)

        total_sq = jax.tree_util.tree_reduce(
            lambda acc, x: acc + leaf_contrib(x),
            grads, 0.0
        )
        return jnp.sqrt(total_sq + eps)

    T = alpha.shape[0]
    return jax.vmap(scaled_norm_i)(jnp.arange(T))


def gradnorm_update(params, batch, alpha, multitask_loss,target_task_loss=1., eps=1e-8):
    """
    params: current model parameters
    batch: your multi-task batch
    alpha: current per-task weighting (shape = (T,))
    alpha_lr: learning rate for alpha
    target_task_loss: e.g. the average or reference scale of each task's loss
    """
    # 1. Evaluate the T losses
    # XLA will likey optimize this to be done in parallel
    losses = multitask_loss(params, *batch)  # shape (T,)

    # 2. Get the T gradients in one backward pass
    grads = jax.jacrev(multitask_loss)(params, *batch)  # PyTree of shape (T, ...)

    # 3. Weighted combined gradient
    weighted_grads = weighted_sum_of_grads(grads, alpha)

    # 4. Compute gradient norms for each task
    # Flatten or do a tree-reduce
    def squared_norm(g):
        return jnp.sum(g**2)
    # shape (T,) of gradient norms
    grad_norms = jax.vmap(lambda i: jax.tree_util.tree_reduce(
        lambda tot, x: tot + squared_norm(x[i]),
        grads, 0.0))(jnp.arange(len(alpha)))
    grad_norms = jnp.sqrt(grad_norms + eps)

    # 5. Compute the GradNorm loss for alpha
    # The idea: We want alpha_i * L_i to have the same ratio as grad_norm_i / average_grad_norm
    # This is the key formula from the paper. For brevity, let's do a simple version:
    avg_loss = jnp.mean(losses)
    loss_ratios = losses / (target_task_loss + eps)
    # The "GradNorm objective" typically looks like: sum | grad_norm_i - (avg_grad_norm * (loss_i / avg_loss)^R ) |
    # We'll do R=1 for a demonstration
    R = 1.0
    avg_grad_norm = jnp.mean(grad_norms)
    target_norms = avg_grad_norm * (loss_ratios ** R)
    gradnorm_obj = jnp.sum(jnp.abs(grad_norms - target_norms))

    # 6. Get gradient wrt alpha
    dalpha = jax.grad(lambda a: gradnorm_obj)(alpha)
    # # Update alpha
    # new_alpha = alpha - alpha_lr * dalpha
    # # Usually we clamp alpha >= 0 or do a softmax
    # new_alpha = jnp.maximum(new_alpha, 0.0)

    # 7. Return final combined gradient for updating params, plus new alpha
    return weighted_grads, dalpha

In [112]:
grad_params, grad_alpha = gradnorm_update(params, (jax.random.key(0),models_mask, thetas, x, bvals, bvecs), jnp.array([0.5,1.]), model.loss_fn)

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)
(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)


In [113]:
grad_alpha

Array([0., 0.], dtype=float32)

In [110]:
grad_params

State({
  'encoder': {
    'initial_layer': {
      'bias': VariableState(
        type=Param,
        value=Array([ 0.036,  2.199, -1.347, -0.356,  0.458, -0.931,  0.912, -1.245,
               -0.282, -2.573,  1.491, -1.452, -0.598,  0.796,  1.77 ,  0.186,
               -0.062, -0.439,  0.141, -0.268,  0.588,  1.184, -1.399, -0.301,
                1.211,  1.252,  0.598, -1.384, -1.059,  1.159,  1.899,  1.276,
                1.393, -0.665,  1.057, -0.099, -0.549,  2.352, -0.03 , -0.493,
               -1.669,  0.538, -1.568,  1.045,  2.026, -0.023,  0.321, -0.707,
               -0.931,  0.026,  1.567, -0.046,  0.083, -1.104,  0.13 , -0.768,
               -0.653,  1.311, -1.704,  0.064, -3.004, -1.38 , -0.026,  0.045],      dtype=float32)
      ),
      'kernel': VariableState(
        type=Param,
        value=Array([[ 0.   ,  0.001, -0.   , -0.   ,  0.   , -0.   ,  0.   , -0.   ,
                -0.   , -0.001,  0.   , -0.   , -0.   ,  0.   ,  0.   ,  0.   ,
                -0. 